In [17]:
!pip install duckdb
!pip install dataretrieval


In [18]:

import duckdb

path = "data/bronze/washington_stream_sites_validated.parquet"

# 1. Create a persistent connection
con = duckdb.connect()

# 2. Register the Parquet file as a virtual table/view
con.execute(f"CREATE VIEW stream_sites AS SELECT * FROM read_parquet('{path}')")

# 3. Now you can run pure, clean SQL anywhere in your notebook!
q = f"""
SELECT * FROM stream_sites
"""

# 4. Execute the query and fetch results as a DataFrame
df = con.execute(q).fetchdf()
print(df.head())

  OrganizationIdentifier           OrganizationFormalName  \
0                USGS-ID  USGS Idaho Water Science Center   
1                USGS-ID  USGS Idaho Water Science Center   
2                USGS-ID  USGS Idaho Water Science Center   
3                USGS-ID  USGS Idaho Water Science Center   
4                USGS-ID  USGS Idaho Water Science Center   

  MonitoringLocationIdentifier                      MonitoringLocationName  \
0                USGS-12392892  BLANCHARD CR AT NORTH FORK NR BLANCHARD ID   
1                USGS-12395501   AUX GAGE FOR PEND OREILLE R AT NEWPORT WA   
2                USGS-12423675                   ROCK CREEK NR ROCKFORD WA   
3                USGS-13334301          SNAKE RIVER NR ANATONE WA NEW GAGE   
4                USGS-13335250  SNAKE RIVER TRIBUTARY NO. 8 NR LEWISTON ID   

  MonitoringLocationTypeName MonitoringLocationDescriptionText  \
0                     Stream                               NaN   
1                     Stream    

# Fetch one site worth of data

I've selected the site USGS-12422000 [(monitoring site here)](https://staging.waterdata.usgs.gov/monitoring-location/USGS-12422000/statistics/#selectedDataTypes=daily-00060-0) for a test, as I have determined that it has:
* recent data, within the 120-day period between 01-01-2026 and 05-01-2026
* It has daily data
* Discrete sample data
  
 | Data type | Frequency | Data data range |
 | --- | --- | --- |
 | Discharge, cubic feet per second | Daily | 1948-12-01 - 2026-05-05|

## Verify that the data is available

I want to verify we get the same data no matter what method we use. I'm going to test:
* curl, 
* the REST API, and
* the dataretrieval package
  
## Verify data prescence with curl


Using the documentation at [www.waterqualitydata.us](https://www.waterqualitydata.us/webservices_documentation/#queries-using-post), we can see an example curl command:

``` bash
# for results in csv
curl -X POST --header 'Content-Type: application/json' -d '{"siteid":["USGS-12422000"]} \ ' 'https://www.waterqualitydata.us/data/Result/search?mimeType=csv&zip=no' --output ./site_USGS-12422000_curl_results.csv

# or in zip:
curl -X POST --header 'Content-Type: application/json' --header 'Accept: application/zip' -d '{"siteid":["USGS-12422000"]} \ ' 'https://www.waterqualitydata.us/data/Result/search?mimeType=csv&zip=yes' --output ./site_USGS-12422000_curl_results.zip

# The zip file is substantially smaller for this call, at 8.2k vs 221k.
# Prefer the zip method for less abuse of the API.
224K -rw-r--r--  1 woodm woodm 221K May  8 12:32 site_USGS-12422000_curl_results.csv
 12K -rw-r--r--  1 woodm woodm 8.2K May  8 12:32 site_USGS-12422000_curl_results.zip

```



## Verify data prescence with Python requests lib using REST API

I can do the same thing in Python with requests, as shown below. 

In [ ]:
import requests
import zipfile
import pandas as pd
import os

def get_absolute_url(url_params: dict) -> str:
    """ Fill in the base URL with the provided query parameters to create the full URL for the API request. """
    
    # Validate the query_target parameter to ensure it's either 'sites' or 'results', which determines the endpoint we will hit.
    # I guess they're technically called "Stations" in the API, but my brain said "use sites instead". 
    # TODO: Decide on a consistent naming convention for this parameter to avoid confusion in the future.
    if url_params.get("query_target") not in ["sites", "results"]:
        raise ValueError(f"Invalid query_target: {url_params.get('query_target')}. Expected 'sites' or 'results'.")
    
    query_target = "Station" if url_params.get("query_target") == "sites" else "Result"
    
    # Join our URL parameters into a query string, excluding the 'query_target' since it's used to determine the endpoint rather than being a query parameter itself.
    query_string = '&'.join([f"{key}={value}" for key, value in url_params.items() if key != "query_target"])
    
    full_url = f"https://www.waterqualitydata.us/data/{query_target}/search?{query_string}"
    
    return full_url

def get_filename_from_response(url_params: dict, response: requests.Response) -> str:
    """ Parse relevant details from the URL or response headers to create a meaningful filename for the downloaded data. """
    # For simplicity, we can just use a fixed filename here, but in the real implementation 
    # I'd want the filename to reflect the query parameters (e.g., include the site_id or date range).
    # That just makes it easier to manage multiple files if we run multiple queries.
    
    site_id = url_params.get('siteid', 'unknown').replace('-', '_') # ensure filename is safe by replacing hyphens with underscores

    query_target = url_params.get('query_target', 'unknown')
    
    # if its a zip, we use the .zip extension, otherwise we can infer the extension from the mimeType parameter or default to .csv
    if url_params.get('zip', 'no').lower() == 'yes':
        file_extension = 'zip'
    else:
        file_extension = url_params.get('mimeType', 'unknown').lower()

    return f"site_{site_id}_dataretrieval_{query_target}.{file_extension}"

def download_data(query_url_params: dict) -> str:
    """ Main function to handle the data download process. It constructs the URL, makes the API request, and saves the response content to a file. Returns the filename of the saved data. """
    url = get_absolute_url(query_url_params)
    
    headers = {
        'Content-Type': 'application/json',
        'Accept': 'application/zip' if query_url_params.get('zip', 'no').lower() == 'yes' else 'text/csv',
    }
    
    response = requests.post(url, headers=headers, json=query_url_params.get("payload", {}))

    if response.status_code == 200:
        filename = get_filename_from_response(query_url_params, response)
        
        with open(filename, 'wb') as f:
            f.write(response.content)
        return filename
    else:
        print(f"Request failed with status code: {response.status_code}")
        print(f"Error: {response.text}")
        return None

query_url_params = {
    "query_target": "results",
    "mimeType": "csv",
    "zip": "yes",
    "payload":  {
        'siteid': [
            'USGS-12422000'
        ],
    }
}

result = download_data(query_url_params)

if not result:
    print("Data download failed.")
else:
    print(f"Data downloaded and saved as: {result}")
    
    # unzip the file if it's a zip
    if result.endswith('.zip'):

        with zipfile.ZipFile(result, 'r') as zip_ref:
            zip_ref.extractall("data/raw/")
        print(f"Extracted {result} to data/raw/")
        

# The extracted file is named 'data/raw/result.csv', so we can read it into a DataFrame to verify it looks correct.

extracted_file_path = "data/raw/result.csv"
try:
    df_request = pd.read_csv(extracted_file_path)
    print("Extracted data preview:")
    print(df_request.head())
    
    # Rename to "site_{site_id}_{method}_results.csv" to compare with the file we get from the WQP API & curl
    print("Renamed extracted file to: data/raw/site_USGS-12422000_requests_results.csv")
    os.rename(extracted_file_path, "data/raw/site_USGS-12422000_requests_results.csv")
except FileNotFoundError:
    print("Extracted file not found.")
except Exception as e:
    print(f"Error occurred while reading the extracted file: {e}")  


Data downloaded and saved as: site_unknown_dataretrieval_results.zip
Extracted site_unknown_dataretrieval_results.zip to data/raw/
Extracted data preview:
  OrganizationIdentifier                OrganizationFormalName  \
0                USGS-WA  USGS Washington Water Science Center   
1                USGS-WA  USGS Washington Water Science Center   
2                USGS-WA  USGS Washington Water Science Center   
3                USGS-WA  USGS Washington Water Science Center   
4                USGS-WA  USGS Washington Water Science Center   

   ActivityIdentifier ActivityTypeCode ActivityMediaName  \
0  nwiswa.01.97801257   Sample-Routine             Water   
1  nwiswa.01.97801257   Sample-Routine             Water   
2  nwiswa.01.97801257   Sample-Routine             Water   
3  nwiswa.01.97801257   Sample-Routine             Water   
4  nwiswa.01.97801257   Sample-Routine             Water   

  ActivityMediaSubdivisionName ActivityStartDate ActivityStartTime/Time  \
0           

## And now we try the same retrieval using dataretrieval.wqp

In [ ]:
from dataretrieval import wqp
import pandas as pd

# [site of interes](https://staging.waterdata.usgs.gov/monitoring-location/USGS-12422000/#dataTypeId=continuous-00065--2127022798&period=P7D&showFieldMeasurements=true)
# I verified that it has data within the last 120 days.
site_id = 'USGS-12422000'

# Call the WQP API
df_wqp, result_metadata = wqp.get_results(
    siteid=site_id
)

df_wqp.to_csv("data/raw/site_USGS-12422000_wqp_results.csv", index=False)


## Now load all three and compare

Pandas has a built-in tool to test if dataframes are mathematically identical. Honestyly this is the first time I've ever used it. 

In [74]:
import pandas as pd

df_curl = pd.read_csv("data/raw/site_USGS-12422000_curl_results.csv")
df_requests = pd.read_csv("data/raw/site_USGS-12422000_requests_results.csv")
df_wqp = pd.read_csv("data/raw/site_USGS-12422000_wqp_results.csv")

# Sort them all by a unique identifier so the row order matches perfectly
sort_cols = ['OrganizationIdentifier', 'MonitoringLocationIdentifier']

df_curl_sorted = df_curl.sort_values(by=sort_cols).reset_index(drop=True)
df_requests_sorted = df_requests.sort_values(by=sort_cols).reset_index(drop=True)
df_wqp_sorted = df_wqp.sort_values(by=sort_cols).reset_index(drop=True)

try:
    # check_dtype=False prevents failures just because one is float32 and the other is float64
    # check_like=True allows columns to be in different orders
    
    # To check all three, check both pairs: curl vs requests, and curl vs wqp. If both of those match, then by transitive property requests and wqp must also match.
    pd.testing.assert_frame_equal(
        df_curl_sorted,
        df_requests_sorted,
        check_dtype=False,
        check_like=True
    )
    
    pd.testing.assert_frame_equal(
        df_curl_sorted,
        df_wqp_sorted,
        check_dtype=False,
        check_like=True
    )

    print("SUCCESS: The Requests API pull via curl, requests, and the WQP package pull are mathematically identical!")
except AssertionError as e:
    print(f"MISMATCH FOUND:\n{e}")

SUCCESS: The Requests API pull via curl, requests, and the WQP package pull are mathematically identical!


## Successful pull, now what

Now that we've got proof that we can pull the same data from all three, verify we have the full gamut.

On the monitoring website, we have the following data about the discrete sample data available:

```
Discrete sample data
55 Observed properties (data types) available - with data from 1978-05-11 to 2000-04-05
23 Sampling activities
```

In [76]:
df_wqp.info()

# save to parquet for future use. We'll likely be naming this a very different name in the future when we have more than one site, but for now this is fine.
df_wqp.to_parquet("data/bronze/site_USGS-12422000_wqp_results.parquet", index=False)

<class 'pandas.DataFrame'>
RangeIndex: 414 entries, 0 to 413
Data columns (total 63 columns):
 #   Column                                             Non-Null Count  Dtype  
---  ------                                             --------------  -----  
 0   OrganizationIdentifier                             414 non-null    str    
 1   OrganizationFormalName                             414 non-null    str    
 2   ActivityIdentifier                                 414 non-null    str    
 3   ActivityTypeCode                                   414 non-null    str    
 4   ActivityMediaName                                  414 non-null    str    
 5   ActivityMediaSubdivisionName                       414 non-null    str    
 6   ActivityStartDate                                  414 non-null    str    
 7   ActivityStartTime/Time                             414 non-null    str    
 8   ActivityStartTime/TimeZoneCode                     414 non-null    str    
 9   ActivityEndDate      

## Columns of interest

Okay, there are a lot of columns here to explore. lets start by verifying the summary on the monitoring site:

Data should contain:
 * Only the chosen site id
 * 414 rows
 * 55 Observed properties (`CharacteristicName`?)
 * data range from from 1978-05-11 to 2000-04-05 (`ActivityStartDate`?)
 * 21 sampling activities (`ActivityIdentifier`?)

In [83]:
import duckdb

path = "data/bronze/site_USGS-12422000_wqp_results.parquet"

# 1. Create a persistent connection
con = duckdb.connect()

# 2. Register the Parquet file as a virtual table/view
con.execute(f"CREATE VIEW stream_results AS SELECT * FROM read_parquet('{path}')")

# query the DISTINCT combinations of OrganizationIdentifier and MonitoringLocationIdentifier to verify we have the expected number of unique sites
query = """
SELECT 
    DISTINCT OrganizationIdentifier, MonitoringLocationIdentifier
FROM
    stream_results
"""

df = con.execute(query).fetchdf()
assert len(df) == 1, f"Expected 1 unique site, but found {len(df)}. Unique sites found:\n{df}"
assert df.iloc[0]['MonitoringLocationIdentifier'] == 'USGS-12422000', f"Expected site ID 'USGS-12422000', but found '{df.iloc[0]['MonitoringLocationIdentifier']}'"
print("SUCCESS: The WQP results data contains exactly 1 unique site as expected.")

SUCCESS: The WQP results data contains exactly 1 unique site as expected.


In [ ]:
# query the number of rows to verify we have the expected number of results
query = """
SELECT COUNT(*) AS num_rows
FROM stream_results
"""

df_count = con.execute(query).fetchdf()
assert df_count['num_rows'][0] == 414, f"Expected 414 rows, but found {df_count['num_rows'][0]}."

In [92]:
# get the date range of the results to verify it matches what we expect based on our query parameters
query = """
SELECT 
    MIN(TRY_CAST(ActivityStartDate AS DATE)) AS min_date,
    MAX(TRY_CAST(ActivityStartDate AS DATE)) AS max_date
FROM stream_results
"""

df_dates = con.execute(query).fetchdf()
print(df_dates)

# Because TRY_CAST returns actual Date objects, Pandas will safely handle this assertion
# Before I figured this out it was disagreeing with me. Check your data types lol
assert df_dates['min_date'][0] >= pd.Timestamp('1978-05-11'), "Minimum date does not match expected"
assert df_dates['max_date'][0] <= pd.Timestamp('2000-04-05'), "Maximum date does not match expected"

    min_date   max_date
0 1978-05-11 2000-04-05


In [93]:
# ActivityIdentifier should return 21 unique values for this site, which matches what we see in the WQP results data. Let's verify that with a SQL query.
query = """
SELECT 
    COUNT(DISTINCT ActivityIdentifier) AS unique_activities
FROM stream_results
"""

df_activities = con.execute(query).fetchdf()
assert df_activities['unique_activities'][0] == 21, f"Expected 21 unique activities, but found {df_activities['unique_activities'][0]}."
print("SUCCESS: The WQP results data contains exactly 21 unique activities as expected.")

AssertionError: Expected 21 unique activities, but found 11.

In [ ]:
# I have tried several ways to find what column represents the "21 sampling activities" we see in the WQP data, and failed so lets brute force it.

df = con.execute("SELECT * FROM stream_results").fetchdf()

# Get unique counts for every column
unique_counts = df.nunique()

# Filter for exactly 21
cols_with_21 = unique_counts[unique_counts == 21]

print(cols_with_21)

ResultAnalyticalMethod/MethodName    21
dtype: int64


## I wish that had been documented better

Of course 21 sampling activities means "ResultAnalyticalMethod/MethodName"...

Anyway, it looks like we have the full dataset, pulled by all three methods. Now we can pull the results data for all site_ids from the stations dataset, before moving into silver for the results as well. 